# ReZeb — Обучение YOLOv11 на датасете строительных дефектов

**Запускать в Google Colab с GPU (T4 или лучше)**

Датасет: [GYU-DET](https://github.com/IamSunday/Multi-defect-type-beam-bridge-dataset-GYU-DET) — 11 123 фото, 6 классов:
- `crack` — трещина
- `spalling` — скол бетона  
- `seepage` — протечка
- `honeycomb` — раковина (пустота в бетоне)
- `exposed_rebar` — обнажённая арматура
- `hole` — отверстие/пустота

Ожидаемое время обучения: ~4-6 часов на T4
Результат: ONNX-модель, загруженная в S3 для ml-service

In [ ]:
# 1. Проверка GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'GPU не найден — переключитесь на GPU Runtime!')

In [ ]:
# 2. Установка зависимостей
!pip install ultralytics roboflow huggingface_hub -q

In [ ]:
# 3. Скачивание GYU-DET датасета
# Вариант А: с GitHub (нужен git lfs)
!git lfs install
!git clone https://github.com/IamSunday/Multi-defect-type-beam-bridge-dataset-GYU-DET.git /content/dataset
import os
print('Файлы датасета:', os.listdir('/content/dataset'))

In [ ]:
# 4. Если GYU-DET недоступен — используем Roboflow конкретный датасет
# Раскомментируйте и добавьте свой API ключ Roboflow

# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
# project = rf.workspace("shm-agftj").project("concrete-defect-detection-pl8ed")
# dataset = project.version(2).download("yolov11")
# DATASET_PATH = dataset.location
# print('Датасет скачан в:', DATASET_PATH)

In [ ]:
# 5. Создаём dataset.yaml если его нет
import yaml, os

DATASET_PATH = '/content/dataset'

dataset_yaml = {
    'path': DATASET_PATH,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 6,
    'names': ['crack', 'spalling', 'seepage', 'honeycomb', 'exposed_rebar', 'hole']
}

yaml_path = f'{DATASET_PATH}/dataset.yaml'
if not os.path.exists(yaml_path):
    with open(yaml_path, 'w') as f:
        yaml.dump(dataset_yaml, f, allow_unicode=True)
    print('Создан dataset.yaml')
else:
    with open(yaml_path) as f:
        print('Существующий dataset.yaml:')
        print(f.read())

In [ ]:
# 6. Обучение YOLOv11s
from ultralytics import YOLO

model = YOLO('yolo11s.pt')  # Small модель — баланс скорость/качество

results = model.train(
    data=f'{DATASET_PATH}/dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,           # GPU
    workers=2,
    patience=20,        # Early stopping
    save=True,
    save_period=10,
    val=True,
    plots=True,
    project='/content/runs',
    name='rezeb_defects',
    # Аугментации для строительных фото
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    degrees=15.0,       # Ротация — стройфото под разными углами
    translate=0.1,
    scale=0.5,
    mosaic=1.0,
    mixup=0.1,
)

print('Метрики обучения:')
print(results)

In [ ]:
# 7. Оценка на тестовом сплите
best_model = YOLO('/content/runs/rezeb_defects/weights/best.pt')
metrics = best_model.val(data=f'{DATASET_PATH}/dataset.yaml', split='test')
print(f'mAP@0.5: {metrics.box.map50:.3f}')
print(f'mAP@0.5-0.95: {metrics.box.map:.3f}')
print(f'Precision: {metrics.box.mp:.3f}')
print(f'Recall: {metrics.box.mr:.3f}')

In [ ]:
# 8. Экспорт в ONNX
best_model.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    opset=17,
    dynamic=False,
)
print('ONNX модель сохранена в /content/runs/rezeb_defects/weights/best.onnx')

In [ ]:
# 9. Загрузка в S3 (MinIO или реальный S3)
import boto3
from pathlib import Path

# Заполните ваши данные
S3_ENDPOINT = 'https://storage.yandexcloud.net'  # или ваш MinIO
S3_ACCESS_KEY = 'YOUR_ACCESS_KEY'
S3_SECRET_KEY = 'YOUR_SECRET_KEY'
S3_BUCKET = 'rezeb-models'
MODEL_KEY = 'yolov11_construction_defects.onnx'

s3 = boto3.client(
    's3',
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=S3_ACCESS_KEY,
    aws_secret_access_key=S3_SECRET_KEY,
)

onnx_path = '/content/runs/rezeb_defects/weights/best.onnx'
s3.upload_file(onnx_path, S3_BUCKET, MODEL_KEY)
print(f'Модель загружена: s3://{S3_BUCKET}/{MODEL_KEY}')

In [ ]:
# 10. Тест инференса на примере фото
from ultralytics import YOLO
import requests
from PIL import Image
from io import BytesIO

# Тестовое фото — трещина в бетоне
test_url = 'https://upload.wikimedia.org/wikipedia/commons/4/4e/Concrete_crack.jpg'
resp = requests.get(test_url)
img = Image.open(BytesIO(resp.content))

onnx_model = YOLO('/content/runs/rezeb_defects/weights/best.onnx', task='detect')
results = onnx_model(img, conf=0.45)

for r in results:
    print('Обнаружено:', len(r.boxes), 'объектов')
    for box in r.boxes:
        cls = onnx_model.names[int(box.cls)]
        conf = float(box.conf)
        print(f'  {cls}: {conf:.2f}')
    r.show()

## После обучения

1. Скачайте `/content/runs/rezeb_defects/weights/best.onnx` локально
2. Загрузите в S3 (ячейка 9)
3. Обновите `ml_service/main.py` — замените Roboflow API на локальный ONNX инференс:

```python
import onnxruntime as ort
import numpy as np
from PIL import Image

session = ort.InferenceSession('yolov11_construction_defects.onnx')
# ... preprocessing + postprocessing
```

4. Ожидаемые метрики на GYU-DET:
   - mAP@0.5 ≥ 0.55 (целевая из ТЗ: ≥ 0.50 для критических дефектов)
   - Recall ≥ 0.80 для `crack` и `exposed_rebar`

## Если метрики низкие

- Добавьте свои фото с российских строек (Phase 3.1 чеклиста)
- Увеличьте `epochs` до 200
- Попробуйте `yolo11m.pt` (medium) вместо small
- Добавьте специфичные классы: `кирпичная_кладка_нарушение`, `сварной_шов_дефект`